In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from google.colab import files

print("Silakan upload file CSV dataset...")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename, sep=";")

NUMERIC_FEATURES = [
    "Age", "Academic Pressure", "Work Pressure", "CGPA",
    "Study Satisfaction", "Job Satisfaction", "Work/Study Hours", "Financial Stress"
]
CATEGORICAL_FEATURES = [
    "Gender", "Sleep Duration", "Dietary Habits",
    "Have you ever had suicidal thoughts ?", "Family History of Mental Illness"
]
TARGET = "Depression"

df = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET]].dropna()


for col in CATEGORICAL_FEATURES:
    unique_vals = sorted(df[col].astype(str).unique())
    mapping = {v: i for i, v in enumerate(unique_vals)}
    df[col] = df[col].astype(str).map(mapping)

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].values.astype(float)
y = df[TARGET].values.astype(int)


x_min = X.min(axis=0)
x_max = X.max(axis=0)
diff  = x_max - x_min
diff[diff == 0] = 1

X = (X - x_min) / diff


np.random.seed(42)
idx     = np.random.permutation(len(X))
X_train = X[idx[:500]]
y_train = y[idx[:500]]
X_test  = X[idx[500:700]]
y_test  = y[idx[500:700]]

print("=" * 45)
print("DATASET INFO")
print("=" * 45)
print(f"  Total data      : {len(X)}")
print(f"  Training samples: {len(X_train)}")
print(f"  Test samples    : {len(X_test)}")
print(f"  Jumlah fitur    : {X.shape[1]}")
print(f"  Kelas train     : 0={sum(y_train==0)}, 1={sum(y_train==1)}")
print(f"  Kelas test      : 0={sum(y_test==0)}, 1={sum(y_test==1)}")
print(f"  Scaling         : Normalisasi Min-Max [0, 1]")



def euclidean(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

def minkowski(x1, x2, p=3):
    return np.power(np.sum(np.abs(x1 - x2) ** p), 1/p)

def mahalanobis(x1, x2, inv_cov):
    diff = x1 - x2
    return np.sqrt(np.dot(np.dot(diff.T, inv_cov), diff))



def knn_predict(X_train, y_train, X_test, k, distance_type, p=3):
    predictions = []

    if distance_type == "mahalanobis":
        cov     = np.cov(X_train.T)
        inv_cov = np.linalg.pinv(cov)

    for x in X_test:
        distances = []

        for i, x_train in enumerate(X_train):
            if distance_type == "euclidean":
                d = euclidean(x, x_train)
            elif distance_type == "minkowski":
                d = minkowski(x, x_train, p)
            elif distance_type == "mahalanobis":
                d = mahalanobis(x, x_train, inv_cov)

            distances.append((d, y_train[i]))

        distances.sort(key=lambda x: x[0])
        k_neighbors = distances[:k]
        labels      = [label for _, label in k_neighbors]
        most_common = Counter(labels).most_common(1)[0][0]
        predictions.append(most_common)

    return np.array(predictions)



def evaluate(y_test, y_pred, title):
    TP = np.sum((y_pred == 1) & (y_test == 1))
    TN = np.sum((y_pred == 0) & (y_test == 0))
    FP = np.sum((y_pred == 1) & (y_test == 0))
    FN = np.sum((y_pred == 0) & (y_test == 1))

    accuracy  = (TP + TN) / len(y_test)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\n=== {title} ===")
    print(f"  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  Confusion Matrix:")
    print(f"              Pred 0   Pred 1")
    print(f"  Actual 0  :  {TN:>5}    {FP:>5}")
    print(f"  Actual 1  :  {FN:>5}    {TP:>5}")

    return accuracy



K = 5
print(f"\n\nMenjalankan KNN (K={K})...\n")

y_pred_euc = knn_predict(X_train, y_train, X_test, K, "euclidean")
acc_euc    = evaluate(y_test, y_pred_euc, "Euclidean")

y_pred_min = knn_predict(X_train, y_train, X_test, K, "minkowski", p=3)
acc_min    = evaluate(y_test, y_pred_min, "Minkowski (p=3)")

y_pred_mah = knn_predict(X_train, y_train, X_test, K, "mahalanobis")
acc_mah    = evaluate(y_test, y_pred_mah, "Mahalanobis")



results = {
    "Euclidean"      : acc_euc,
    "Minkowski (p=3)": acc_min,
    "Mahalanobis"    : acc_mah,
}

print("\n" + "=" * 45)
print("RINGKASAN PERBANDINGAN AKURASI (K=5)")
print("=" * 45)
for name, acc in results.items():
    bar  = int(acc * 30)
    best = " <- terbaik" if acc == max(results.values()) else ""
    print(f"  {name:<20}: {acc*100:.2f}%  {bar}{best}")
print("=" * 45)

Silakan upload file CSV dataset...


Saving Student Depression Dataset (1).csv to Student Depression Dataset (1) (3).csv
DATASET INFO
  Total data      : 27898
  Training samples: 500
  Test samples    : 200
  Jumlah fitur    : 13
  Kelas train     : 0=216, 1=284
  Kelas test      : 0=88, 1=112
  Scaling         : Normalisasi Min-Max [0, 1]


Menjalankan KNN (K=5)...


=== Euclidean ===
  Accuracy  : 0.8050  (80.50%)
  Precision : 0.8067
  Recall    : 0.8571
  F1-Score  : 0.8312
  Confusion Matrix:
              Pred 0   Pred 1
  Actual 0  :     65       23
  Actual 1  :     16       96

=== Minkowski (p=3) ===
  Accuracy  : 0.8000  (80.00%)
  Precision : 0.8051
  Recall    : 0.8482
  F1-Score  : 0.8261
  Confusion Matrix:
              Pred 0   Pred 1
  Actual 0  :     65       23
  Actual 1  :     17       95

=== Mahalanobis ===
  Accuracy  : 0.7600  (76.00%)
  Precision : 0.7623
  Recall    : 0.8304
  F1-Score  : 0.7949
  Confusion Matrix:
              Pred 0   Pred 1
  Actual 0  :     59       29
  Actual 1  :     1

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=5, random_state=99)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
acc_dt = evaluate(y_test, y_pred_dt, "Decision Tree")


=== Decision Tree ===
  Accuracy  : 0.7500  (75.00%)
  Precision : 0.8100
  Recall    : 0.7232
  F1-Score  : 0.7642
  Confusion Matrix:
              Pred 0   Pred 1
  Actual 0  :     69       19
  Actual 1  :     31       81


In [ ]:
results = {
    "Euclidean"        : acc_euc,
    "Minkowski (p=3)"  : acc_min,
    "Mahalanobis"      : acc_mah,
    "Decision Tree"    : acc_dt,
}

print("RINGKASAN PERBANDINGAN AKURASI")

best_acc = max(results.values())

for name, acc in results.items():

    if acc == best_acc:
        best = "  TERBAIK"
    else:
        best = ""

    print(f"{name:<20} : {acc*100:.2f}%{best}")


RINGKASAN PERBANDINGAN AKURASI
Euclidean            : 80.50%  TERBAIK
Minkowski (p=3)      : 80.00%
Mahalanobis          : 76.00%
Decision Tree        : 75.00%
